In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [2]:
dataset =pd.read_csv("CKD.csv")
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [3]:
indep1 = dataset.drop(columns=['classification'])

In [4]:
dep = dataset['classification']

In [5]:
indep = pd.get_dummies(indep1, drop_first = True)

In [7]:
dataset['classification'].value_counts()

classification
yes    249
no     150
Name: count, dtype: int64

In [15]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split (indep, dep, test_size =1/3, random_state = 0)

In [17]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [19]:
from sklearn.svm import SVC

In [20]:
from sklearn.model_selection import GridSearchCV
param_grid =  {'kernel': ['rbf', 'sigmoid', 'linear'],
              'C':[10,100,1000,2000,3000], 'gamma':['auto', 'scale']}
grid = GridSearchCV(SVC(probability = True), param_grid, refit = True, cv=5, verbose = 3, n_jobs = -1, scoring ="f1_weighted")
grid.fit(x_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,estimator,SVC(probability=True)
,param_grid,"{'C': [10, 100, ...], 'gamma': ['auto', 'scale'], 'kernel': ['rbf', 'sigmoid', ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


In [21]:
re = grid.cv_results_
grid_predictions = grid.predict(x_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

print(cm)
print(clf_report)

[[51  0]
 [ 1 81]]
              precision    recall  f1-score   support

          no       0.98      1.00      0.99        51
         yes       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [22]:
from sklearn.metrics import f1_score
f1_macro = f1_score(y_test, grid_predictions, average = 'weighted')
print(" the f1_macro value for best parameter {}:". format(grid.best_params_), f1_macro)

 the f1_macro value for best parameter {'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}: 0.9924946382275899


In [23]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])

1.0

In [24]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.022447,0.014661,0.020844,0.014545,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,1.000000,1.000000,0.988859,0.014751,3
1,0.017329,0.002081,0.015557,0.003233,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.981569,1.000000,1.000000,0.981031,1.000000,0.992520,0.009163,1
2,0.012975,0.001369,0.013488,0.002054,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",0.981569,0.962636,0.962573,0.981031,0.981217,0.973805,0.009147,13
3,0.019604,0.002787,0.018159,0.001387,10,scale,rbf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,1.000000,1.000000,0.988859,0.014751,3
4,0.016889,0.001023,0.016725,0.001754,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.981569,1.000000,1.000000,0.981031,1.000000,0.992520,0.009163,1
5,0.016309,0.002165,0.015496,0.001385,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.981569,0.962636,0.962573,0.981031,0.981217,0.973805,0.009147,13
6,0.015735,0.003210,0.014509,0.002379,100,auto,rbf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,0.981031,1.000000,0.985066,0.013807,5
7,0.014476,0.001853,0.014958,0.001484,100,auto,sigmoid,"{'C': 100, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.981569,0.943699,0.944023,0.981031,1.000000,0.970065,0.022459,23
8,0.017044,0.001020,0.016571,0.002087,100,auto,linear,"{'C': 100, 'gamma': 'auto', 'kernel': 'linear'}",0.981569,0.962636,0.962573,0.981031,0.981217,0.973805,0.009147,13
9,0.018723,0.003473,0.015888,0.001621,100,scale,rbf,"{'C': 100, 'gamma': 'scale', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,0.981031,1.000000,0.985066,0.013807,5
